# OpenAI Top 5 News Items

This notebook loads `OPENAI_API_KEY` from `.env`, uses OpenAI's hosted `web_search` tool, and returns the top 5 news items for a query.

It does not print your API key.

## 1. Install packages if needed

In [ ]:
# %pip install openai python-dotenv pydantic

## 2. Load OpenAI API key

In [46]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not found. Add it to .env first.")

client = OpenAI()
MODEL = "gpt-4.1-mini"


class NewsItem(BaseModel):
    title: str
    summary: str
    source_name: str
    source_url: str

class ValidatedNewsItem(NewsItem):
    url_validity: bool

class NewsResult(BaseModel):
    items: list[NewsItem]

print("OpenAI API key loaded.")

OpenAI API key loaded.


## 3. Ask OpenAI to return a Python list

Change `NEWS_QUERY` to search a different topic.

In [ ]:
NEWS_QUERY = "videogame"

# the output "response" is an openai sdk object
# type(response.output_parsed) = NewsResult
# type(response.output_parsed.items) = list. It is a list of NewsItem object 
response = client.responses.parse(
    model=MODEL,
    tools=[{"type": "web_search"}],
    tool_choice="required",
    input=f"""
Use web search to find exactly 5 current news articles for this query: {NEWS_QUERY}

Requirements:
- Return only specific news article pages.
- The URL must point directly to the individual article, not a homepage, category page, tag page, search page, or news index.
- Prefer recent, reputable sources.
- Do not include duplicate stories from different URLs unless they add meaningfully different reporting.
- If a source URL looks like a section page, such as `/ai/news`, `/technology`, `/category/artificial-intelligence`, or a site homepage, reject it and find another result.

Good URL example:
https://www.tomsguide.com/ai/anthropic-abruptly-disables-fable-5-and-mythos-5-following-us-government-order

Bad URL example:
https://www.tomsguide.com/ai/news

Return exactly 5 items.

Only return URLs that are likely to be live and reachable.
Do not return URLs that appear to be broken, archived, placeholder, paywall redirect-only, or 404 pages.
""",
    text_format=NewsResult,
)



In [45]:
# Convert the parsed Pydantic objects into a normal Python list of dictionaries.
news_items = [item.model_dump() for item in response.output_parsed.items]

news_items

[{'title': 'Valve Targeted by Dutch Class Action Over Videogame Pricing',
  'summary': 'A Dutch consumer rights foundation has initiated a class action against Valve, alleging that the company is abusing its dominant position and charging excessive commissions that inflate prices for consumers.',
  'source_name': 'MLex',
  'source_url': 'https://www.mlex.com/mlex/articles/2488418/valve-targeted-by-dutch-class-action-over-videogame-pricing'},
 {'title': "Frontier Developments' FY26 Revenue Rises on CMS Strength",
  'summary': 'Videogame maker Frontier Developments reported a rise in fiscal year 2026 revenue, driven by the strength of its CMS (Content Management System) offerings.',
  'source_name': 'TradingView News',
  'source_url': 'https://www.tradingview.com/news/reuters.com%2C2026%3Anewsml_L1N42I063%3A0-videogame-maker-frontier-developments-fy26-revenue-rises-on-cms-strength/'},
 {'title': 'Haunted Videogame No Players Online Resurrected on Steam After DMCA Takedown',
  'summary': 

In [47]:
import requests

def url_is_live(url):
    """
    Return True if the URL seems reachable.
    Return False if the URL is broken, blocked, timed out, or returns an error status.
    """
    try:
        # First try a HEAD request.
        # HEAD asks the server whether the page exists without downloading the full page body.
        # This is faster and lighter than GET.
        response = requests.head(
            url,
            allow_redirects=True,  
            timeout=10,            
            headers={
                # Some websites block script-like requests.
                # This User-Agent makes the request look more like a normal browser.
                "User-Agent": "Mozilla/5.0"
            },
        )

        # Some websites do not allow HEAD requests.
        # 403 means forbidden.
        # 405 means method not allowed.
        # In those cases, the page may still work in a browser, so try GET.
        if response.status_code in [403, 405]:
            response = requests.get(
                url,
                allow_redirects=True,
                timeout=10,
                headers={"User-Agent": "Mozilla/5.0"},
                stream=True,  # Do not download the whole page body immediately.
            )

        # HTTP status codes below 400 usually mean success or redirect.
        # Examples:
        # 200 = OK
        # 301/302 = redirect
        #
        # Status codes 400 and above usually mean failure.
        # Examples:
        # 404 = not found
        # 500 = server error
        return response.status_code < 400

    except requests.RequestException:
        # If the request fails because of timeout, bad URL, SSL error,
        # connection error, or too many redirects, treat the URL as not live.
        return False

In [48]:
validated_items = [
    ValidatedNewsItem(
        **item.model_dump(),
        url_validity=url_is_live(item.source_url),
    )
    for item in response.output_parsed.items
]

validated_items

[ValidatedNewsItem(title='Valve Targeted by Dutch Class Action Over Videogame Pricing', summary='A Dutch consumer rights foundation has initiated a class action against Valve, alleging that the company is abusing its dominant position and charging excessive commissions that inflate prices for consumers.', source_name='MLex', source_url='https://www.mlex.com/mlex/articles/2488418/valve-targeted-by-dutch-class-action-over-videogame-pricing', url_validity=True),
 ValidatedNewsItem(title="Frontier Developments' FY26 Revenue Rises on CMS Strength", summary='Videogame maker Frontier Developments reported a rise in fiscal year 2026 revenue, driven by the strength of its CMS (Content Management System) offerings.', source_name='TradingView News', source_url='https://www.tradingview.com/news/reuters.com%2C2026%3Anewsml_L1N42I063%3A0-videogame-maker-frontier-developments-fy26-revenue-rises-on-cms-strength/', url_validity=True),
 ValidatedNewsItem(title='Haunted Videogame No Players Online Resurr

## 4. Optional: inspect metadata

In [38]:
print("Response ID:", response.id)
print("Usage:", response.usage)

Response ID: resp_06d4a869b6ef2246006a2f934ed474819bab742516c3c6b38b
Usage: ResponseUsage(input_tokens=8794, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=420, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=9214)
